# Feature Importance Check (no retraining)

This notebook re-does only the **feature engineering + train/test split** from `FE_main.ipynb` (cheap, no model fitting) and then **loads your already-saved models** from `models_fe/*.pkl` to run `permutation_importance`.

It answers: *does the model actually use the FinBERT `emb_*` columns, or is it dominated by the macro/price features?*

**Before running:** place this notebook in the same folder as `FE_main.ipynb` (so `dataset/`, `fed_speech_embeddings.npy`, and `models_fe/` are all reachable with the same relative paths). If your saved models live in a folder called `models/` instead of `models_fe/`, just change `MODEL_DIR` below.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, roc_auc_score

warnings.filterwarnings('ignore')

# ---- change this if your saved models are in a different folder ----
MODEL_DIR = "models_fe"

## 1. Rebuild the merged DataFrame
(identical to `FE_main.ipynb` section 1 — no model fitting happens here, just pandas)

In [2]:
embeddings = np.load("fed_speech_embeddings.npy")

fed_speech = pd.read_csv("dataset/fed_speech.csv")
dates = pd.to_datetime(fed_speech["date"])

prices = pd.read_csv("dataset/price_action.csv")
prices["date"] = pd.to_datetime(prices["date"])

In [3]:
fed_speech['embedding'] = list(embeddings)
fed_speech['date'] = pd.to_datetime(fed_speech['date'])

In [4]:
macro = pd.read_csv('dataset/macro_indicators.csv')

macro['date'] = pd.to_datetime(macro['date'])

macro = macro.set_index('date').sort_index()
macro["unemployment"] = macro["unemployment"].shift(30)
macro["growth_rate"]  = macro["growth_rate"].shift(30)

daily_index = pd.date_range(start=macro.index.min(), end=macro.index.max(), freq='D')

macro_daily = macro.reindex(daily_index).ffill().reset_index()
macro_daily = macro_daily.rename(columns={'index': 'date'})

In [5]:
merged_df = pd.merge(prices, macro_daily, on="date")

In [6]:
target_front = ['date', 'unemployment', 'interest_rate', 'growth_rate',
                'SPX','TNX','GOLD','VIX','DXY'] 
all_cols = merged_df.columns.tolist()
remaining_cols = [c for c in all_cols if c not in target_front]

merged_df = merged_df[target_front + remaining_cols]

In [7]:
for col in ['SPX','GOLD','TNX','DXY','VIX']:
    merged_df[f'{col}_ret'] = np.log(
        merged_df[col] / merged_df[col].shift(1)
    )

for col in ['SPX', 'GOLD', 'TNX', 'DXY', 'VIX']:
    merged_df[f'{col}_mom_3'] = (
        merged_df[col].shift(1) / merged_df[col].shift(4) - 1
    )
    merged_df[f'{col}_mom_7'] = (
        merged_df[col].shift(1) / merged_df[col].shift(8) - 1
    )
    merged_df[f'{col}_mom_30'] = (
        merged_df[col].shift(1) / merged_df[col].shift(31) - 1
    )

    merged_df[f'{col}_t-3'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(3)
        .mean()
    )
    merged_df[f'{col}_t-7'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(7)
        .mean()
    )
    merged_df[f'{col}_t-30'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(30)
        .mean()
    )

    merged_df[f'{col}_vol_7'] = (
        merged_df[f'{col}_ret'].shift(1).rolling(7).std()
    )
    merged_df[f'{col}_vol_30'] = (
        merged_df[f'{col}_ret'].shift(1).rolling(30).std()
    )

for col in ['SPX','GOLD','TNX','DXY','VIX']:
    merged_df[f'{col}_t+3'] = (
        merged_df[col].shift(-4) /
        merged_df[col].shift(-1)
    ) - 1

    merged_df[f'{col}_t+7'] = (
        merged_df[col].shift(-8) /
        merged_df[col].shift(-1)
    ) - 1

    merged_df[f'{col}_t+30'] = (
        merged_df[col].shift(-31) /
        merged_df[col].shift(-1)
    ) - 1
    
lag_cols = [
    col for col in fed_speech.columns
    if "_t-" in col or
       "unemployment" in col or
       "fed interest rate" in col or
       "growth rate" in col
]

fed_speech[lag_cols] = fed_speech[lag_cols].shift(1)

merged_df = merged_df.drop(columns=[
    'SPX_ret','GOLD_ret','TNX_ret','DXY_ret','VIX_ret',
    'SPX','GOLD','TNX','DXY','VIX'
])

In [8]:
fed_speech = fed_speech.merge(merged_df, on='date', how='left')

In [9]:
fed_speech = fed_speech.drop(columns=['content'])

In [10]:
emb_matrix = np.vstack(fed_speech["embedding"].values)

emb_df = pd.DataFrame(
    emb_matrix,
    index=fed_speech.index,
    columns=[f"emb_{i}" for i in range(emb_matrix.shape[1])]
)

fed_speech = pd.concat(
    [fed_speech.drop(columns=["embedding"]), emb_df],
    axis=1
)

## 2. Rebuild the train/test split
(identical to `FE_main.ipynb` section 2A — still no fitting, just splitting)

In [11]:
fed_speech = fed_speech.sort_values("date").reset_index(drop=True)
fed_speech["date"] = pd.to_datetime(fed_speech["date"])

In [12]:
def get_weight(speaker):
    speaker = speaker.lower()
    
    if "chair" in speaker and "vice" not in speaker:
        return 3
    elif "vice chair" in speaker or "vice chairman" in speaker:
        return 2
    elif "governor" in speaker:
        return 1
    else:
        return 0.5

fed_speech["sample_weight"] = fed_speech["speaker"].apply(get_weight)

In [13]:
split_date = "2023-01-01"

train_df = fed_speech[fed_speech["date"] < split_date]
test_df  = fed_speech[(fed_speech["date"] >= split_date) & (fed_speech["date"] <= '2025-12-31')]

In [14]:
target_cols = [
    "SPX_t+3", "SPX_t+7", "SPX_t+30",
    "GOLD_t+3","GOLD_t+7","GOLD_t+30",
    "VIX_t+3","VIX_t+7","VIX_t+30",
    "TNX_t+3","TNX_t+7","TNX_t+30",
]

drop_cols = ["date", "title", "speaker", "sample_weight", "id"] + \
            target_cols + \
            ["DXY_t+3","DXY_t+7","DXY_t+30"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target_cols]
w_train = train_df["sample_weight"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df[target_cols]

print(f"X_test shape: {X_test.shape}")

X_test shape: (334, 811)


## 3. Load the already-trained models (no fitting)
Loads `{col}.pkl` for every target from `MODEL_DIR`, plus `feature_columns.pkl` as a sanity check that the feature-engineering above reproduced the exact same column set/order the models were trained on.

In [15]:
feature_columns = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))

missing = set(feature_columns) - set(X_test.columns)
extra   = set(X_test.columns) - set(feature_columns)
assert not missing, f"X_test is missing columns the model expects: {missing}"
if extra:
    print(f"Note: dropping {len(extra)} extra column(s) not seen at train time: {extra}")

# Reorder to match exactly what the models were trained on
X_test = X_test[feature_columns]

models = {}
for col in target_cols:
    models[col] = joblib.load(os.path.join(MODEL_DIR, f"{col}.pkl"))
    print(f"✓ loaded {col}")

✓ loaded SPX_t+3
✓ loaded SPX_t+7
✓ loaded SPX_t+30
✓ loaded GOLD_t+3
✓ loaded GOLD_t+7
✓ loaded GOLD_t+30
✓ loaded VIX_t+3
✓ loaded VIX_t+7
✓ loaded VIX_t+30
✓ loaded TNX_t+3
✓ loaded TNX_t+7
✓ loaded TNX_t+30


## 4. Permutation importance per target
For each of the 12 models: shuffle each feature (10 repeats), measure the drop in R², and sum the drop across all 768 `emb_*` columns vs everything else. This is the definitive check for whether the model is actually using the FinBERT text embedding, or ignoring it in favor of the macro/price features.

In [16]:
emb_cols   = [c for c in feature_columns if c.startswith("emb_")]
other_cols = [c for c in feature_columns if not c.startswith("emb_")]
emb_idx    = [feature_columns.index(c) for c in emb_cols]

N_REPEATS = 10
RANDOM_STATE = 0

results_summary = []

for col in target_cols:
    model = models[col]
    y_true = y_test[col]

    # drop rows with NaN target (t+30 horizon runs off the end of the date range)
    valid = y_true.notna()
    X_valid = X_test[valid]
    y_valid = y_true[valid]

    result = permutation_importance(
        model, X_valid, y_valid,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring="neg_mean_squared_error",
    )

    importances = np.clip(result.importances_mean, a_min=0, a_max=None)
    total_importance = importances.sum()
    emb_importance = importances[emb_idx].sum()
    emb_share = emb_importance / total_importance if total_importance > 0 else float("nan")

    results_summary.append({
        "target": col,
        "n_valid_rows": int(valid.sum()),
        "total_importance": total_importance,
        "emb_importance": emb_importance,
        "emb_share_pct": emb_share * 100,
    })

    print(f"{col:<10}  n={int(valid.sum()):<4}  embedding share: {emb_share:.2%}")

summary_df = pd.DataFrame(results_summary)
summary_df

SPX_t+3     n=334   embedding share: 27.61%
SPX_t+7     n=334   embedding share: 40.45%
SPX_t+30    n=334   embedding share: 28.74%
GOLD_t+3    n=334   embedding share: 98.86%
GOLD_t+7    n=334   embedding share: 34.10%
GOLD_t+30   n=334   embedding share: 18.09%
VIX_t+3     n=334   embedding share: 25.52%
VIX_t+7     n=334   embedding share: 3.78%
VIX_t+30    n=334   embedding share: 5.02%
TNX_t+3     n=334   embedding share: 82.82%
TNX_t+7     n=334   embedding share: 71.46%
TNX_t+30    n=334   embedding share: 53.07%


,target,n_valid_rows,total_importance,emb_importance,emb_share_pct
0,SPX_t+3,334,0.000003,9.477007e-07,27.614651
1,SPX_t+7,334,0.000097,3.923996e-05,40.451773
2,SPX_t+30,334,0.000518,1.488822e-04,28.736194
3,GOLD_t+3,334,0.000006,6.122441e-06,98.858953
4,GOLD_t+7,334,0.000017,5.837678e-06,34.102922
5,GOLD_t+30,334,0.000578,1.045470e-04,18.090100
6,VIX_t+3,334,0.000239,6.108865e-05,25.523202
7,VIX_t+7,334,0.000783,2.959894e-05,3.781025
8,VIX_t+30,334,0.013329,6.690832e-04,5.019837
9,TNX_t+3,334,0.000153,1.271042e-04,82.815414


## 5. Overall verdict

In [17]:
overall_emb_share = summary_df["emb_importance"].sum() / summary_df["total_importance"].sum()

print(f"Overall embedding importance share across all 12 targets: {overall_emb_share:.2%}\n")

if overall_emb_share < 0.02:
    print("→ Near zero. The trees are essentially ignoring the FinBERT embedding — "
          "this is a model-level fact (it never learned to use the text), not a "
          "'wrong date' issue. Changing the test date won't fix the bearish-speech-stays-bullish behavior.")
else:
    print("→ The embedding does carry measurable weight. If a specific speech/date pair "
          "still doesn't flip direction, that's more likely a 'macro signal dominates on that "
          "particular day' issue than the model ignoring text entirely — worth also trying the "
          "extreme-embedding-perturbation test and the historical-crash-date test discussed earlier.")

Overall embedding importance share across all 12 targets: 15.98%

→ The embedding does carry measurable weight. If a specific speech/date pair still doesn't flip direction, that's more likely a 'macro signal dominates on that particular day' issue than the model ignoring text entirely — worth also trying the extreme-embedding-perturbation test and the historical-crash-date test discussed earlier.


In [1]:
# ═══════════════════════════════════════════════════════════════
# ABLATION: refit each target WITHOUT embedding columns
# Uses the SAME tuned hyperparameters as the original HGB models —
# this isn't a full re-tune, just isolates whether embeddings
# are contributing signal or noise with the architecture held fixed.
# ═══════════════════════════════════════════════════════════════

emb_cols_all = [c for c in X_train.columns if c.startswith("emb_")]
non_emb_cols = [c for c in X_train.columns if not c.startswith("emb_")]

X_train_noemb = X_train[non_emb_cols]
X_test_noemb  = X_test[non_emb_cols]

print(f"Dropped {len(emb_cols_all)} embedding cols, kept {len(non_emb_cols)}")

def get_metrics(true_series, pred_series):
    auc = roc_auc_score((true_series > 0).astype(int), pred_series)
    acc = (np.sign(pred_series) == np.sign(true_series)).mean()
    rmse = np.sqrt(mean_squared_error(true_series, pred_series))
    return auc, acc, rmse

noemb_models = {}
ablation_rows = []

for col in target_cols:
    y_train_col = y_train[col]
    valid_train = y_train_col.notna()
    X_tr = X_train_noemb[valid_train]
    y_tr = y_train_col[valid_train]
    w_tr = w_train[valid_train]

    # reuse the tuned params found WITH embeddings — same architecture,
    # different feature set, so any AUC change isolates the embeddings' effect
    final_params = {
        **best_params[col],
        'max_bins':            255,
        'early_stopping':      True,
        'n_iter_no_change':    50,
        'validation_fraction': 0.15,
        'random_state':        42
    }

    model = HistGradientBoostingRegressor(**final_params)
    model.fit(X_tr, y_tr, sample_weight=w_tr)
    noemb_models[col] = model

    y_true = y_test[col]
    valid_test = y_true.notna()
    y_pred = model.predict(X_test_noemb[valid_test])
    y_true_valid = y_true[valid_test]

    auc, acc, rmse = get_metrics(y_true_valid, y_pred)
    ablation_rows.append({"target": col, "auc": auc, "acc": acc, "rmse": rmse})
    print(f"✓ {col:<12} no-emb AUC={auc:.3f}  acc={acc:.3f}  rmse={rmse:.4f}")

noemb_results = pd.DataFrame(ablation_rows).set_index("target")

# ═══════════════════════════════════════════════════════════════
# COMPARE: with-embeddings (hgb_results, from earlier) vs no-embeddings
# ═══════════════════════════════════════════════════════════════
with_emb_auc = {col: roc_auc_score(
                    (y_test[col][y_test[col].notna()] > 0).astype(int),
                    models[col].predict(X_test[y_test[col].notna()])
                ) for col in target_cols}

comparison = pd.DataFrame({
    "auc_with_emb": with_emb_auc,
    "auc_no_emb":   noemb_results["auc"],
})
comparison["delta"] = comparison["auc_no_emb"] - comparison["auc_with_emb"]
comparison["emb_share_pct"] = summary_df.set_index("target")["emb_share_pct"]

print("\n=== With-Embeddings vs No-Embeddings AUC ===")
print(comparison.round(3).sort_values("emb_share_pct"))

NameError: name 'X_train' is not defined